In [1]:
import pypeline
import numpy as np
from glob import glob
import os
import sys
import mne_bids
from contextlib import contextmanager
import mne
from datetime import datetime
%load_ext autoreload

%autoreload 2

In [2]:
EXPERIMENT_NAME = "banyan"  # name of the experiment
parent_dir = "../raw_data"  # directory where your raw data (folder containing brainvision, eyetracking asc, and behavior is stored)
data_dir = "../data"  # where to output data
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
file_prefix = ""  # prefix to your vhdr files. Assuming it is in the format [prefix][number]

overwrite_subs = False  # if you want to overwrite the data for a subject, set to True
subject_dirs = [
    "001",
    "002",
    "003",
    "004",
    "005",
    "006",
    "007",
    "008",
    "009",
    "010",
    #"011", # preprocessed in a different script due to eye sampling rate issues
    "012",
    "013",
    "014",
    "015",
    "016",
    "017",
    "018",
    "019",
    "020",
    "021",
    "022",
    "023",
    "024",
    "025",
    "026",
    "027",
    "028",
    "029",
    "030",
    "031",
    "032",
    "033",
    "034",
]  # if you want to analyze a specific subset of subjects
if len(subject_dirs) == 0:
    subject_dirs = sorted(glob("*", root_dir=parent_dir))
if not overwrite_subs:
    subject_dirs = [sub for sub in subject_dirs if sub not in [f[4:] for f in glob("sub-*", root_dir=data_dir)]]

print("Subject to analyze: ", subject_dirs)


TRIAL_START_TIME = -0.45  # epoch start before your designated timelock code
TRIAL_END_TIME = 1.4
BASELINE_TIME = (-0.35, 0)  # time for baseline correction

SRATE = 500  # hz, will resample if different from the raw data srate
FILTER_FREQS = (None, 50)  # None to not do one of the filtering steps

LINEAR_R2 = 0.3

event_dict = {  # this should be a dict of names of ALL the event codes that appear
    "trl_start": 1,
    "attn/ss2/abs": 12,
    "attn/ss4/abs": 14,
    "attn/ss6/abs": 16,
    "attn/ss8/abs": 18,
    "attn/ss2/placeholder": 22,
    "attn/ss4/placeholder": 24,
    "attn/ss6/placeholder": 26,
    "attn/ss8/placeholder": 28,
    "wm/ss2/abs": 32,
    "wm/ss4/abs": 34,
    "wm/ss6/abs": 36,
    "wm/ss8/abs": 38,
    "wm/ss2/placeholder": 42,
    "wm/ss4/placeholder": 44,
    "wm/ss6/placeholder": 46,
    "wm/ss8/placeholder": 48,
    "delay_start": 2,
    "attention_probe": 3,  # if probe attention, the code seq is 2 3 2 4
    "delay_end": 4,
    "response": 5,
}
# event_dict is a list of name: number pairings for all the TRIAL event codes
# event_code_dict: a dict of code: sequence pairings for each trial
# so, if you have a trial with fixation (1) -> SS2 stimulus (12) -> delay (3) -> test (4), that you want to map to code  12:
# {12 : [1,12,3,4]}... and so on
event_code_dict = {}  # define event codes based on sequence

for key, ev in event_dict.copy().items():
    if ev > 10:
        event_dict.update({key + "/PROBE": ev + 1})  # add in keys for targets
        event_code_dict.update({ev: [1, ev, 2, 4, 4]})
        event_code_dict.update({ev + 1: [1, ev, 2, 3, 2, 4, 4]})


POSITION_TO_TIMELOCK = 1  # which position (IN THE LIST ABOVE) to timelock to. Should be the one which indicates the condition.

EYEKEYWORD = "SYNC"

# edge case when we forgot to start the recording, manually drop certain trials
# Must be in the form of {'subject number':[list of ints]}
EEG_TRIALS_DROP = {}
EYE_TRIALS_DROP = {"010": [0, 1, 2, 3]}  # edge case when we forgot to start the recording, drop certain trials
# channels to ignore when rejecting trials. Will also be excluded from analysis.
SUB_IGNORE_CHANS = {}

NO_EYES = []  # subjects with no eye data
DROP_CHANNELS = (
    []
)  # channels to delete from the dataset entirely. Recommendation is to leave this blank and instead set REJ_TRIALS_IGNORE later on

Subject to analyze:  ['001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '012', '013', '014', '015', '016', '017', '018', '019', '020', '021', '022', '023', '024', '025', '026', '027', '028', '029', '030', '031', '032', '033', '034']


In [3]:
pre = pypeline.Preprocess(
    data_dir=data_dir,
    root_dir=parent_dir,
    experiment_name=EXPERIMENT_NAME,
    srate=SRATE,
    trial_start=TRIAL_START_TIME,
    trial_end=TRIAL_END_TIME,
    event_dict=event_dict,
    event_code_dict=event_code_dict,
    timelock_ix=POSITION_TO_TIMELOCK,
    baseline_time=BASELINE_TIME,
    rejection_time=None,
    reject_between_codes=[1, 4],
    no_et_spaces=False,
    drop_channels=DROP_CHANNELS,
    filter_freqs=FILTER_FREQS,
)

In [4]:
@contextmanager
def redirect_stdout(new_stdout):  # writes the output to a log file
    save_stdout = sys.stdout
    save_stderr = sys.stderr
    sys.stdout = new_stdout
    sys.stderr = sys.stdout
    try:
        yield None
    finally:
        sys.stdout = save_stdout
        sys.stderr = save_stderr


# with open('preprocessing_log.txt','a+') as f:
#     with redirect_stdout(f):

print("\n\n\n##########################\n" + "STARTING PREPROCESSING RUN\n" + "##########################\n\n\n")
print(f'Run started at {datetime.now().strftime("%H:%M:%S")}')
for subject_number in subject_dirs:

    print(
        "\n\n#############################\n"
        + f"## STARTING NEW SUBJECT {subject_number} ##\n"
        + "#############################\n"
    )

    #####################
    #### IMPORT DATA ####
    #####################

    # import into the RAW bids dataset
    eeg, eeg_events = pre.import_eeg(subject_number, overwrite=True)
    reref_index = mne.pick_channels(eeg.ch_names, ["TP9"])
    eeg.load_data().apply_function(
        pre.rereference_to_average, picks=["eeg"], reref_values=np.squeeze(eeg.get_data()[reref_index])
    )
    eeg.filter(*pre.filter_freqs, n_jobs=-1)

    pre.import_behavior(subject_number)

    ########################################
    #### PREPROCESS EEG AND MAKE EPOCHS ####
    ########################################

    if subject_number in NO_EYES:
        epochs = pre.make_eeg_epochs(eeg, eeg_events, eeg_trials_drop=EEG_TRIALS_DROP.get(subject_number, None))
    else:
        eye, eye_events = pre.import_eyetracker(subject_number, keyword=EYEKEYWORD)

        epochs = pre.make_and_sync_epochs(
            eeg,
            eeg_events,
            eye,
            eye_events,
            eeg_trials_drop=EEG_TRIALS_DROP.get(subject_number, None),
            eye_trials_drop=EYE_TRIALS_DROP.get(subject_number, None),
        )

    ###############################
    #### DO ARTIFACT REJECTION ####
    ###############################

    p2p = pre.artreject_slidingP2P(
        epochs, rejection_criteria={"eeg": 100e-6, "eog": 200}, win=200, win_step=100
    )  # peak to peak in the window
    saccades = pre.artreject_step(
        epochs, rejection_criteria={"eyegaze": pre.deg2pix(0.5), "eog": 50}, win=80, win_step=10
    )  # saccades in EOG or eye tracking
    steps = pre.artreject_step(
        epochs, rejection_criteria={"eeg": 60e-6}, win=250, win_step=20
    )  # steps (saccade like) in EEG

    absolute_value = pre.artreject_value(
        epochs, rejection_criteria={"eyegaze": pre.deg2pix(1), "eeg": 100e-6, "eog": 300}
    )  # absolute value rejection
    linear_fit = pre.artreject_linear(epochs)  # linear fit (drift) rejection
    flatline = pre.artreject_flatline(
        epochs,
        rejection_criteria={"eeg": 0, "eog": 0, "eyegaze": 0},
        flatline_duration=200,
    )  # check for flatlines

    # combine rejection reasons
    rej_electrodes = p2p | saccades | steps | absolute_value | linear_fit | flatline
    rej_reasons = np.char.array(
        np.full(rej_electrodes.shape, "", dtype="<U30")
    )  # NOTE: dtype is important, must be >= the max possible str length
    rej_reasons[p2p] = "P2P "
    rej_reasons[saccades] = rej_reasons[saccades] + "SAC "
    rej_reasons[steps] = rej_reasons[steps] + "STEP "
    rej_reasons[absolute_value] = rej_reasons[absolute_value] + "ABS "
    rej_reasons[linear_fit] = rej_reasons[linear_fit] + "LIN "
    rej_reasons[flatline] = rej_reasons[flatline] + "FLAT "

    rej_counts = lambda x: f"{x.any(1).sum()} ({round(x.any(1).sum() / x.shape[0] * 100,1)}%)"
    print(
        (
            f"Rejected {rej_electrodes.any(1).sum()} trials ({round(rej_electrodes.any(1).sum() / rej_electrodes.shape[0] * 100,1)}%) for the following reasons:\n"
            f"Peak to peak amplitude: {rej_counts(p2p)}\n"
            f"Saccades: {rej_counts(saccades)}\n"
            f"Steps: {rej_counts(steps)}\n"
            f"Absolute value: {rej_counts(absolute_value)}\n"
            f"Linear fit: {rej_counts(linear_fit)}\n"
            f"Flatline: {rej_counts(flatline)}\n"
        )
    )

    print(
        "Worst electrodes by count:\n"
        + "\n".join(
            [f"{epochs.ch_names[i]}: {rej_electrodes[:,i].sum()}" for i in np.argsort(rej_electrodes.sum(0))[::-1][0:5]]
        )
    )

    #################################
    #### SAVE DATA AS DERIVATIVE ####
    #################################

    pre.save_all_data(subject_number, epochs, rej_reasons)
print(mne_bids.make_report(data_dir))




##########################
STARTING PREPROCESSING RUN
##########################



Run started at 10:26:49


#############################
## STARTING NEW SUBJECT 001 ##
#############################

Extracting parameters from ../raw_data/001/SCR_001.vhdr...
Setting channel info structure...


/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/preprocess.py:162: RuntimeWarning: Online software filter detected. Using software filter settings and ignoring hardware values
  eegdata = mne.io.read_raw_brainvision(
/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/preprocess.py:162: RuntimeWarning: Channels contain different highpass filters. Lowest (weakest) filter setting (0.00 Hz) will be stored.
  eegdata = mne.io.read_raw_brainvision(
/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/preprocess.py:162: RuntimeWarning: Channels contain different lowpass filters. Highest (weakest) filter setting (500.00 Hz, Nyquist limit) will be stored.
  eegdata = mne.io.read_raw_brainvision(
/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/preprocess.py:162: RuntimeWarning: Not setting positions of 3 eog/misc channels found in montage:
['

Used Annotations descriptions: [np.str_('New Segment/'), np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  3'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S 12'), np.str_('Stimulus/S 14'), np.str_('Stimulus/S 16'), np.str_('Stimulus/S 18'), np.str_('Stimulus/S 22'), np.str_('Stimulus/S 24'), np.str_('Stimulus/S 26'), np.str_('Stimulus/S 28'), np.str_('Stimulus/S 32'), np.str_('Stimulus/S 34'), np.str_('Stimulus/S 36'), np.str_('Stimulus/S 38'), np.str_('Stimulus/S 42'), np.str_('Stimulus/S 44'), np.str_('Stimulus/S 46'), np.str_('Stimulus/S 48')]


/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/preprocess.py:196: RuntimeWarning: Converting data files to BrainVision format
  mne_bids.write_raw_bids(
/opt/homebrew/Caskroom/miniforge/base/envs/preproc_pype/lib/python3.12/site-packages/pybv/io.py:690: UserWarning: Encountered unsupported non-voltage units: C
Note that the BrainVision format specification supports only µV.
  warn(msg)


Writing '../data/sub-001/eeg/sub-001_task-banyan_eeg.json'...
Reading 0 ... 8529679  =      0.000 ...  8529.679 secs...
Filtering raw data in 1 contiguous segment
Setting up low-pass filter at 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 265 samples (0.265 s)



...

/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/preprocess.py:690: RuntimeWarning: The measurement information indicates a low-pass frequency of 500.0 Hz. The decim=2 parameter will result in a sampling frequency of 500.0 Hz, which can cause aliasing artifacts.
  eye_epochs = mne.Epochs(


Rejected 67 trials (4.2%) for the following reasons:
Peak to peak amplitude: 1 (0.1%)
Saccades: 65 (4.1%)
Steps: 0 (0.0%)
Absolute value: 7 (0.4%)
Linear fit: 0 (0.0%)
Flatline: 0 (0.0%)

Worst electrodes by count:
xpos_right: 59
xpos_left: 59
ypos_right: 9
ypos_left: 9
PO8: 1


/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/preprocess.py:1120: RuntimeWarning: This filename (../data/derivatives/sub-034/eeg/sub-034_task-banyan_desc-preprocessed_eeg.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs.save(path.fpath, overwrite=True)


Summarizing participants.tsv ../data/participants.tsv...
Summarizing scans.tsv files [PosixPath('../data/sub-021/sub-021_scans.tsv'), PosixPath('../data/sub-026/sub-026_scans.tsv'), PosixPath('../data/sub-019/sub-019_scans.tsv'), PosixPath('../data/sub-010/sub-010_scans.tsv'), PosixPath('../data/sub-017/sub-017_scans.tsv'), PosixPath('../data/sub-028/sub-028_scans.tsv'), PosixPath('../data/sub-016/sub-016_scans.tsv'), PosixPath('../data/sub-029/sub-029_scans.tsv'), PosixPath('../data/sub-027/sub-027_scans.tsv'), PosixPath('../data/sub-018/sub-018_scans.tsv'), PosixPath('../data/sub-020/sub-020_scans.tsv'), PosixPath('../data/sub-002/sub-002_scans.tsv'), PosixPath('../data/sub-005/sub-005_scans.tsv'), PosixPath('../data/sub-033/sub-033_scans.tsv'), PosixPath('../data/sub-034/sub-034_scans.tsv'), PosixPath('../data/sub-032/sub-032_scans.tsv'), PosixPath('../data/sub-004/sub-004_scans.tsv'), PosixPath('../data/sub-003/sub-003_scans.tsv'), PosixPath('../data/sub-025/sub-025_scans.tsv'), Po

/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/preprocess.py:1122: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  np.save(path.fpath, epochs.get_data())


In [ ]:
%matplotlib qt
sub = input('Enter subject number: ') # you can also just set this to a string

REJ_CHANNELS_IGNORE=['StimTrak', 'HEOG','VEOG','TP9', 'Fp1', 'Fp2'] # exclude 'Fp1' and 'Fp2' here if you want to keep trials where they are noisy, and drop them before analysis later


viz = pypeline.Visualizer(sub,
                            parent_dir = data_dir,
                            load_flags = False,  # we're going to rebuild our rejection flags every time
                            experiment_name=EXPERIMENT_NAME,
                            srate=SRATE,
                            rejection_time=[None, None],
                            downscale={'eyegaze':1e-6,'misc':1,'eeg':1,'eog':1e-6}, # convert to equivalent units (probably uV)
                            channels_drop=['pupil_left','pupil_right'],
                            channels_ignore=REJ_CHANNELS_IGNORE + SUB_IGNORE_CHANS.get(sub, []))


rejection_sums = viz.rej_chans.sum(axis=0)
sort_ix = np.argsort(rejection_sums)[::-1]

for ichan,chan in enumerate(viz.chan_labels[sort_ix]):
    if rejection_sums[sort_ix][ichan] > 0:
        print(chan,rejection_sums[sort_ix][ichan])
viz.preprocess_data_for_plot()
viz.open_figure()

Reading /Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/../data/derivatives/sub-034/eeg/sub-034_task-banyan_desc-preprocessed_eeg.fif ...
    Found the data of interest:
        t =    -450.00 ...    1400.00 ms
        0 CTF compensation matrices available


/Users/henryjones/Documents/research/banyan/Experiment 1/preprocessing_pypeline/pypeline/visualizer.py:56: RuntimeWarning: This filename (../data/derivatives/sub-034/eeg/sub-034_task-banyan_desc-preprocessed_eeg.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  self.epochs_obj = mne.read_epochs(self.data_path.fpath)


Adding metadata with 20 columns
1597 matching events found
No baseline correction applied
0 projection items activated
xpos_left 59
xpos_right 59
ypos_right 9
ypos_left 9
PO8 1
Pz 1


key not recognized: control. Press h for help.
key not recognized: ctrl+alt. Press h for help.
66/1597 trials rejected. Saving annotations as "../data/derivatives/sub-034/eeg/sub-034_task-banyan_desc-preprocessed_rejection_flags.npy"


# Move behavior files

In [ ]:
break

In [4]:
from glob import glob
import pandas as pd
import os

In [6]:
sub_behavior_files = glob(f'../raw_data/*/*_data.csv')
for sbf in sub_behavior_files:
    sub = sbf.split('/')[-2]
    if not os.path.exists(f'../data/derivatives/sub-{sub}/beh'):
        os.makedirs(f'../data/derivatives/sub-{sub}/beh')
    df = pd.read_csv(sbf)
    df.to_csv(f'../data/derivatives/sub-{sub}/beh/sub-{sub}_task-banyan_beh.tsv', sep='\t', index=True)
